# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR^2 Croissant](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library, referencing all data entities by their `@id` fields, as per the Croissant specification.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and instantiate a Croissant dataset object.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset's Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset object
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Let's review available record sets (tables), fields (columns), and their `@id`s.
We will enumerate the record sets and fields available for this dataset.

In [ ]:
# List all available record sets and their fields by @id
print("Available Record Sets in the dataset:")
if hasattr(metadata, "record_sets") and metadata.record_sets:
    for recordset in metadata.record_sets:
        print(f"  - RecordSet @id: {recordset.id} | name: {getattr(recordset, 'name', '')}")
        if hasattr(recordset, 'fields'):
            print("    Fields (columns):")
            for field in recordset.fields:
                print(f"      - Field @id: {field.id} | name: {getattr(field, 'name', '')} | type: {getattr(field, 'data_type', '')}")
else:
    print("(No record sets found in the JSON-LD metadata or not supported by this schema)")

## 3. Data Extraction
We load data from each available record set into a pandas DataFrame. All references to record sets and fields use their Croissant `@id` for clarity and reproducibility.

If record sets are available, they'll be loaded below.

In [ ]:
# Discover all record set @ids
record_sets = []
if hasattr(metadata, "record_sets") and metadata.record_sets:
    record_sets = [r.id for r in metadata.record_sets]
else:
    print("No record sets found -- this dataset may not expose tabular record sets.")

dataframes = {}
for record_set_id in record_sets:
    print(f"Loading records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for {record_set_id} with columns: {df.columns.tolist()}")

# Display the first few rows for the first available record set
if len(record_sets) > 0:
    display_id = record_sets[0]
    print(f"Data preview for RecordSet {display_id}:\n")
    display(dataframes[display_id].head())
else:
    print("No data extracted.")

## 4. Exploratory Data Analysis (EDA)
Filter, normalize, and group numeric fields. 
All operations use fields by their `@id`. (If no record set or suitable numeric field is found, a message will be printed instead.)

In [ ]:
import numpy as np

# Pick a record set and a numeric field by @id for demonstration
record_set_id = None
numeric_field_id = None
group_field_id = None

if hasattr(metadata, "record_sets") and metadata.record_sets:
    for rs in metadata.record_sets:
        if hasattr(rs, "fields") and rs.fields:
            # Find first numeric field in this record set
            for field in rs.fields:
                if getattr(field, "data_type", None) in ["Integer", "Float", "Number"]:
                    record_set_id = rs.id
                    numeric_field_id = field.id
                    # Find another field for grouping, if present and different
                    for group_field in rs.fields:
                        if group_field.id != numeric_field_id:
                            group_field_id = group_field.id
                            break
                    break
            if numeric_field_id:
                break

if record_set_id and numeric_field_id:
    df = dataframes[record_set_id]
    # The mlcroissant library returns field values keyed by their @id
    try:
        # Try to cast values to numeric for computation
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = np.nanmean(df[numeric_field_id])
        filtered_df = df[df[numeric_field_id] > threshold]

        print(f"Filtered records in RecordSet {record_set_id} where field {numeric_field_id} > mean ({threshold:.2f}):")
        display(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by group_field_id if present
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            display(grouped_df.head())
    except Exception as e:
        print(f"Data analysis failed due to: {e}")
else:
    print("No numeric field found in any record set for demonstration.")

## 5. Visualization
Let's plot the distribution of the selected numeric field, and a scatter plot versus a group/category field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and numeric_field_id:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in RecordSet {record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field_id} grouped by {group_field_id}")
        plt.ylabel(numeric_field_id)
        plt.xlabel(group_field_id)
        plt.tight_layout()
        plt.show()
else:
    print("Visualization step skipped: No numeric field available.")

## 6. Conclusion
In this notebook, we've used the `mlcroissant` library to inspect and analyze the FAIR^2 mass spectrometry protein dataset using only Croissant `@id` references for entities:

- Loaded dataset metadata and discovered record sets and their available fields/columns by `@id`.
- Loaded tabular data from each record set (if available).
- Filtered and normalized a numeric field by `@id`, and optionally grouped records for aggregation.
- Visualized numeric field distributions using matplotlib and seaborn.

This approach ensures reproducible, schema-driven FAIR data analysis based on the Croissant specification for scientific datasets.